# 03 — ML Zone Detection

**Project:** ZoneTrend — AI-assisted swing trading  
**Research question:** Can a machine learning model learn to detect supply/demand zones directly from raw candlestick windows, without hand-coded rules?

---

| # | Section | What you see |
|---|---------|-------------|
| 1 | Training Dataset | Window dataset stats, label distribution, sample windows |
| 2 | Walk-Forward CV | Per-class P / R / F1 — zone vs no_zone |
| 3 | Feature Importance | Which candle positions & features drive detection |
| 4 | ML Probability Distribution | Model confidence across all candles |
| 5 | ML Zones on Chart | ML-detected zones plotted on full price history |
| 6 | Rule vs ML Comparison | Side-by-side zone counts, overlap, differences |
| 7 | ML Zone Quality | Distribution of ML zones by type, confidence, time |
| 8 | Active ML Zones | Current zones the model is detecting right now |

**Run first:** `python zone_pipeline.py --skip-detection`

---
## 0 · Setup

In [266]:
import sys, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option('display.float_format', '{:.4f}'.format)

# ── TradingView dark theme ───────────────────────────────────────────────────
PAPER_BG    = '#131722'
PLOT_BG     = '#131722'
GRID_COLOR  = '#1e222d'
TEXT_COLOR  = '#d1d4dc'
WIN_COLOR   = '#26a69a'
LOSS_COLOR  = '#ef5350'
NEUTRAL     = '#b2b5be'
ACCENT      = '#2962ff'
DEMAND_CLR  = 'rgba(38,166,154,0.25)'
SUPPLY_CLR  = 'rgba(239,83,80,0.25)'
DEMAND_LINE = '#26a69a'
SUPPLY_LINE = '#ef5350'
ML_CLR      = 'rgba(41,98,255,0.20)'
ML_LINE     = '#2962ff'
STRUCT_CLR  = {'DBR': WIN_COLOR, 'RBR': '#00bcd4', 'RBD': LOSS_COLOR, 'DBD': '#ff9800'}

def dark_layout(**kw):
    d = dict(
        paper_bgcolor=PAPER_BG, plot_bgcolor=PLOT_BG,
        font=dict(color=TEXT_COLOR, size=12),
        xaxis=dict(gridcolor=GRID_COLOR, zeroline=False),
        yaxis=dict(gridcolor=GRID_COLOR, zeroline=False),
        margin=dict(l=60, r=20, t=50, b=50),
        legend=dict(bgcolor='rgba(0,0,0,0)'),
    )
    d.update(kw)
    return go.Layout(**d)

print(f'Project root: {PROJECT_ROOT}')

# ── Same theme constants as notebook 02 ─────────────────────────────────────
DEMAND_FILL   = 'rgba(38,166,154,0.12)'
DEMAND_LINE   = 'rgba(38,166,154,0.75)'
SUPPLY_FILL   = 'rgba(239,83,80,0.12)'
SUPPLY_LINE   = 'rgba(239,83,80,0.75)'
CHART_THEME   = 'plotly_dark'
PAPER_BG      = '#131722'
PLOT_BG       = '#131722'
GRID_COLOR    = '#1e222d'


Project root: /Users/sachinchoudhary/Desktop/RI/ZoneTrend


In [267]:
with open(PROJECT_ROOT / 'config' / 'config.yaml') as f:
    CFG = yaml.safe_load(f)

SYMBOL = CFG['data']['symbol']
SAFE   = SYMBOL.replace('.', '_').replace('^', 'IDX_')

PROC_PATH      = PROJECT_ROOT / 'data' / 'processed'  / f'{SAFE}.csv'
RULE_ZONES_PATH= PROJECT_ROOT / 'data' / 'zones'      / f'{SAFE}_zones.csv'
ML_ZONES_PATH  = PROJECT_ROOT / 'data' / 'zones'      / f'{SAFE}_ml_zones.csv'
WINDOWS_PATH   = PROJECT_ROOT / 'data' / 'labeled'    / f'{SAFE}_zone_windows_v2.csv'
MODEL_PATH     = PROJECT_ROOT / 'data' / 'models'     / f'{SAFE}_zone_xgb.pkl'

for p in [PROC_PATH, RULE_ZONES_PATH, ML_ZONES_PATH, WINDOWS_PATH, MODEL_PATH]:
    print(f'  {"✓" if p.exists() else "✗ MISSING"}  {p.relative_to(PROJECT_ROOT)}')

  ✓  data/processed/IDX_NSEI.csv
  ✓  data/zones/IDX_NSEI_zones.csv
  ✓  data/zones/IDX_NSEI_ml_zones.csv
  ✓  data/labeled/IDX_NSEI_zone_windows_v2.csv
  ✓  data/models/IDX_NSEI_zone_xgb.pkl


In [268]:
# ── Load all data ────────────────────────────────────────────────────────────
proc       = pd.read_csv(PROC_PATH)
proc['Date'] = pd.to_datetime(proc['Date'])
proc = proc.sort_values('Date').reset_index(drop=True)

rule_zones = pd.read_csv(RULE_ZONES_PATH)
rule_zones['formation_date'] = pd.to_datetime(rule_zones['formation_date'])

ml_zones   = pd.read_csv(ML_ZONES_PATH)
ml_zones['formation_date'] = pd.to_datetime(ml_zones['formation_date'])

windows    = pd.read_csv(WINDOWS_PATH)
windows['date'] = pd.to_datetime(windows['date'])

with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)

print(f'Price candles  : {len(proc)}')
print(f'Rule-based zones: {len(rule_zones)}')
print(f'ML-detected zones: {len(ml_zones)}')
print(f'Training windows : {len(windows)}')
print(f'  Positives (zone): {(windows["label"]==1).sum()}')
print(f'  Negatives (no_zone): {(windows["label"]==0).sum()}')

Price candles  : 2828
Rule-based zones: 100
ML-detected zones: 96
Training windows : 2809
  Positives (zone): 100
  Negatives (no_zone): 2709


---
## 1 · Training Dataset

In [269]:
# ── 1.1  Label distribution ──────────────────────────────────────────────────
n_pos = (windows['label'] == 1).sum()
n_neg = (windows['label'] == 0).sum()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Class Count', 'Class % (log scale)'])

for col, vals, title in [
    (1, [n_neg, n_pos], 'count'),
    (2, [n_neg, n_pos], 'log'),
]:
    fig.add_trace(go.Bar(
        x=['no_zone', 'zone'], y=vals,
        marker_color=[NEUTRAL, WIN_COLOR],
        text=vals, textposition='outside',
        showlegend=False,
    ), row=1, col=col)

fig.update_yaxes(type='log', row=1, col=2)
fig.update_layout(dark_layout(
    title=f'Training Dataset: {len(windows)} windows | '
          f'imbalance {n_neg//max(n_pos,1)}:1 handled via scale_pos_weight',
    height=380,
))
fig.show()
print(f'  no_zone : {n_neg}  ({n_neg/len(windows):.1%})')
print(f'  zone    : {n_pos}  ({n_pos/len(windows):.1%})')
print(f'  scale_pos_weight = {n_neg/max(n_pos,1):.1f}')

  no_zone : 2709  (96.4%)
  zone    : 100  (3.6%)
  scale_pos_weight = 27.1


In [270]:
# ── 1.2  Zone formations over time ──────────────────────────────────────────
zone_windows = windows[windows['label'] == 1].copy()
zone_windows['year'] = zone_windows['date'].dt.year
by_year = zone_windows.groupby('year').size().reset_index(name='n')

fig = go.Figure(go.Bar(
    x=by_year['year'], y=by_year['n'],
    marker_color=WIN_COLOR,
    text=by_year['n'], textposition='outside',
))
fig.update_layout(dark_layout(
    title='Zone Formations per Year (rule-based ground truth)',
    height=360, xaxis_title='Year', yaxis_title='Zones',
))
fig.show()

In [271]:
# ── 1.3  Compute ML probability on all windows ───────────────────────────────
skip_cols = {'label', 'date', 'zone_type'}
feat_cols = [c for c in windows.columns if c not in skip_cols]
X_all     = windows[feat_cols].values.astype(float)
zone_idx  = list(model.classes_).index(1) if hasattr(model, 'classes_') else 1
probs     = model.predict_proba(X_all)[:, zone_idx]
windows['p_zone'] = probs

print(f'P(zone) stats:')
print(f'  Zone windows    : mean={probs[windows["label"]==1].mean():.3f}  '
      f'median={np.median(probs[windows["label"]==1]):.3f}')
print(f'  No-zone windows : mean={probs[windows["label"]==0].mean():.3f}  '
      f'median={np.median(probs[windows["label"]==0]):.3f}')

P(zone) stats:
  Zone windows    : mean=0.989  median=0.989
  No-zone windows : mean=0.010  median=0.001


---
## 2 · Walk-Forward Cross-Validation

In [272]:
# Re-run CV to collect per-class metrics for plotting
from src.models.zone_detection_model import get_X_y, build_model

# Drop p_zone if present — must not leak model output into CV features
windows_clean = windows.drop(columns=['p_zone'], errors='ignore')
X, y, _ = get_X_y(windows_clean)
tscv     = TimeSeriesSplit(n_splits=4)
fold_data = []

for fold_i, (tr, te) in enumerate(tscv.split(X), 1):
    spw   = (y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)
    mdl   = build_model(scale_pos_weight=spw)
    mdl.fit(X[tr], y[tr])
    y_pred = mdl.predict(X[te])
    rep    = classification_report(
        y[te], y_pred,
        target_names=['no_zone','zone'],
        zero_division=0, output_dict=True,
    )
    for cls in ['no_zone', 'zone']:
        fold_data.append({
            'fold': fold_i, 'class': cls,
            'precision': rep[cls]['precision'],
            'recall':    rep[cls]['recall'],
            'f1':        rep[cls]['f1-score'],
            'support':   rep[cls]['support'],
        })

cv_df = pd.DataFrame(fold_data)
print('CV complete.')
print(cv_df.groupby('class')[['precision','recall','f1']].mean().round(3))

CV complete.
         precision  recall     f1
class                            
no_zone     0.9760  0.9810 0.9790
zone        0.2960  0.2520 0.2710


In [273]:
# ── 2.1  F1 heatmap per fold ─────────────────────────────────────────────────
pivot = cv_df.pivot(index='class', columns='fold', values='f1')

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=[f'Fold {c}' for c in pivot.columns],
    y=pivot.index.tolist(),
    colorscale='RdYlGn', zmin=0, zmax=1,
    text=[[f'{v:.2f}' for v in row] for row in pivot.values],
    texttemplate='%{text}',
    colorbar=dict(title='F1'),
))
fig.update_layout(dark_layout(
    title='F1 Score per Class per Walk-Forward Fold  (NOT accuracy)',
    height=300,
))
fig.show()

In [274]:
# ── 2.2  Avg P / R / F1 per class ───────────────────────────────────────────
avg_cls = cv_df.groupby('class')[['precision','recall','f1']].mean().round(3)

fig = go.Figure()
for metric, color in [('precision', WIN_COLOR), ('recall', '#00bcd4'), ('f1', ACCENT)]:
    fig.add_trace(go.Bar(
        name=metric.capitalize(),
        x=avg_cls.index,
        y=avg_cls[metric],
        marker_color=color,
        text=[f'{v:.3f}' for v in avg_cls[metric]],
        textposition='outside',
    ))

fig.add_hline(y=0.5, line_dash='dot', line_color=NEUTRAL, annotation_text='0.5')
fig.update_layout(dark_layout(
    title='Avg Precision / Recall / F1 — Walk-Forward CV',
    height=400, barmode='group',
    yaxis=dict(range=[0, 1.15], gridcolor=GRID_COLOR),
))
fig.show()

---
## 3 · Feature Importance

In [275]:
fi = pd.Series(
    model.feature_importances_,
    index=feat_cols,
).sort_values(ascending=False)

# ── 3.1  Top 30 features ────────────────────────────────────────────────────
top = fi.head(30).sort_values(ascending=True)

fig = go.Figure(go.Bar(
    x=top.values, y=top.index, orientation='h',
    marker_color=[WIN_COLOR if v >= top.median() else NEUTRAL for v in top.values],
    text=[f'{v:.4f}' for v in top.values], textposition='outside',
))
fig.update_layout(dark_layout(
    title='Feature Importance — Top 30 (XGBoost on 20-candle windows)',
    height=700,
    margin=dict(l=220, r=80, t=50, b=40),
    xaxis_title='Importance',
))
fig.show()

In [276]:
# ── 3.2  Importance by candle position (which window position matters most?) ─
import re
pos_imp = {}
for name, imp in fi.items():
    m = re.match(r'c([+-]?\d+)_', name)
    if m:
        pos = int(m.group(1))
        pos_imp[pos] = pos_imp.get(pos, 0) + imp

pos_df = pd.DataFrame(sorted(pos_imp.items()), columns=['position','importance'])

fig = go.Figure(go.Bar(
    x=pos_df['position'], y=pos_df['importance'],
    marker_color=[WIN_COLOR if p >= -2 else NEUTRAL for p in pos_df['position']],
))
fig.add_vline(x=-0.5, line_dash='dash', line_color=LOSS_COLOR,
              annotation_text='departure candle', annotation_position='top right')
fig.update_layout(dark_layout(
    title='Total Feature Importance by Candle Position in Window\n'
          '(position 0 = departure candle, negative = candles before)',
    height=380,
    xaxis_title='Position (0 = most recent)',
    yaxis_title='Sum of feature importances',
))
fig.show()
print('\nTop 5 most important positions:')
print(pos_df.sort_values('importance', ascending=False).head(5).to_string(index=False))


Top 5 most important positions:
 position  importance
        0      0.2605
       -1      0.0845
       -5      0.0494
      -18      0.0480
       -6      0.0475


---
## 4 · ML Probability Distribution

In [277]:
# ── 4.1  P(zone) histogram: zone vs no_zone windows ─────────────────────────
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=windows.loc[windows['label']==1, 'p_zone'],
    name='Zone windows', nbinsx=25,
    marker_color=WIN_COLOR, opacity=0.8,
))
fig.add_trace(go.Histogram(
    x=windows.loc[windows['label']==0, 'p_zone'].sample(300, random_state=42),
    name='No-zone sample (300)', nbinsx=25,
    marker_color=LOSS_COLOR, opacity=0.8,
))
fig.add_vline(x=0.45, line_dash='dash', line_color=ACCENT,
              annotation_text='threshold=0.45')
fig.update_layout(dark_layout(
    title='P(zone) Distribution — Model Confidence on Zone vs No-Zone Windows',
    height=380, barmode='overlay',
    xaxis_title='P(zone)', yaxis_title='Count',
))
fig.show()

In [278]:
# ── 4.2  P(zone) across time on the full price history ───────────────────────
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.55, 0.45], vertical_spacing=0.04,
    subplot_titles=['Price (Close)', 'P(zone) — Model Confidence'])

fig.add_trace(go.Scatter(
    x=proc['Date'], y=proc['Close'],
    mode='lines', name='Close',
    line=dict(color=NEUTRAL, width=1),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=windows['date'], y=windows['p_zone'],
    mode='lines', name='P(zone)',
    line=dict(color=ACCENT, width=1),
    fill='tozeroy', fillcolor='rgba(41,98,255,0.1)',
), row=2, col=1)

# Mark actual zone formation dates
zone_dates = windows[windows['label']==1]
fig.add_trace(go.Scatter(
    x=zone_dates['date'], y=zone_dates['p_zone'],
    mode='markers', name='Actual zone',
    marker=dict(color=WIN_COLOR, size=7, symbol='triangle-up'),
), row=2, col=1)

fig.add_hline(y=0.45, row=2, col=1, line_dash='dash', line_color=LOSS_COLOR,
              annotation_text='threshold')
fig.update_layout(dark_layout(
    title='P(zone) Signal Over Time — Spikes indicate zone formations',
    height=520,
))
fig.show()

---
## 5 · ML-Detected Zones on Full Price History

Interactive candlestick chart with all ML-detected zone bands.
- **Green bands** = demand zones  |  **Red bands** = supply zones
- **Solid border** = active zone  |  **Dashed border** = invalidated
- Hover over a zone band for zone ID, type, ML probability, and formation date


In [279]:
# ── Prepare date-indexed proc and zone_id-indexed ml_zones ───────────────────
# (mirrors notebook 02 — plot functions expect DatetimeIndex on proc)
proc_idx = proc.copy()
proc_idx['Date'] = pd.to_datetime(proc_idx['Date'])
proc_idx = proc_idx.set_index('Date')
proc_idx.index = proc_idx.index.tz_localize(None)

ml_zones_idx = ml_zones.copy()
ml_zones_idx['formation_date'] = pd.to_datetime(ml_zones_idx['formation_date'])
if 'zone_id' in ml_zones_idx.columns:
    ml_zones_idx = ml_zones_idx.set_index('zone_id')

# Compute width_atr for ML zones (used in hover + detail chart)
ml_zones_idx['width_atr'] = ml_zones_idx['width'] / ml_zones_idx['avg_atr'].replace(0, float('nan'))

print(f'proc_idx      : {len(proc_idx)} rows  index={proc_idx.index.dtype}')
print(f'ml_zones_idx  : {len(ml_zones_idx)} zones  index={ml_zones_idx.index.name}')


proc_idx      : 2828 rows  index=datetime64[ns]
ml_zones_idx  : 96 zones  index=zone_id


In [280]:
def plot_zones_on_candles(
    proc,
    zones,
    title='',
    n_candles=None,
    show_emas=True,
):
    """
    Broker-style interactive candlestick chart with ML zone overlays.
    Identical to notebook 02 — adapted hover text for ML zones (ml_prob).
    """
    df = proc.tail(n_candles).copy() if n_candles else proc.copy()
    date_min = df.index.min()
    date_max = df.index.max()

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.78, 0.22], vertical_spacing=0.02,
    )

    # Candlesticks
    fig.add_trace(go.Candlestick(
        x=df.index,
        open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'],
        name='Price',
        increasing=dict(line=dict(color='#26a69a', width=1), fillcolor='#26a69a'),
        decreasing=dict(line=dict(color='#ef5350', width=1), fillcolor='#ef5350'),
        whiskerwidth=0.5, hoverinfo='x+y',
    ), row=1, col=1)

    # Volume bars
    vol_colors = ['#26a69a' if c >= o else '#ef5350'
                  for c, o in zip(df['Close'], df['Open'])]
    fig.add_trace(go.Bar(
        x=df.index, y=df['Volume'].replace(0, None),
        marker_color=vol_colors, name='Volume',
        opacity=0.55, showlegend=False,
        hovertemplate='Vol: %{y:,.0f}<extra></extra>',
    ), row=2, col=1)

    # EMAs
    if show_emas:
        for period, color, width in [(20,'#ff9800',1.0),(50,'#2196F3',1.2),(200,'#ce93d8',1.6)]:
            col_name = f'EMA{period}'
            if col_name in df.columns:
                fig.add_trace(go.Scatter(
                    x=df.index, y=df[col_name], name=f'EMA {period}',
                    line=dict(color=color, width=width),
                    hovertemplate=f'EMA{period}: %{{y:.2f}}<extra></extra>',
                ), row=1, col=1)

    # Zone bands
    for zone_id, z in zones.iterrows():
        fdate     = pd.Timestamp(z['formation_date'])
        is_demand = z['type'] == 'demand'
        status    = z.get('status', 'active')
        is_live   = status in ('active', 'tested')

        inv = z.get('invalidation_date')
        if pd.notna(inv) and pd.Timestamp(inv) < date_min:
            continue
        if fdate > date_max:
            continue

        fill = DEMAND_FILL if is_demand else SUPPLY_FILL
        line = DEMAND_LINE if is_demand else SUPPLY_LINE
        dash = 'solid'    if is_live   else 'dot'
        x0   = max(fdate, date_min)
        x1   = date_max
        prob  = z.get('ml_prob', float('nan'))
        w_atr = z.get('width_atr', z['width'] / max(z.get('avg_atr', 1), 1e-9))

        fig.add_shape(
            type='rect', xref='x', yref='y',
            x0=x0, x1=x1, y0=float(z['bottom']), y1=float(z['top']),
            fillcolor=fill, line=dict(color=line, width=1, dash=dash),
            layer='below',
        )

        hover_txt = (
            f"<b>{zone_id}</b>  {'🟢 Demand' if is_demand else '🔴 Supply'}<br>"
            f"Top: <b>{z['top']:.2f}</b>  Bottom: <b>{z['bottom']:.2f}</b><br>"
            f"Width: {w_atr:.2f}× ATR<br>"
            f"ML Prob: <b>{prob:.3f}</b><br>"
            f"Status: <b>{status}</b>  |  Tests: {int(z.get('test_count', 0))}<br>"
            f"Formed: {fdate.strftime('%d %b %Y')}"
            "<extra></extra>"
        )
        fig.add_trace(go.Scatter(
            x=[x0, x1, x1, x0, x0],
            y=[z['bottom'], z['bottom'], z['top'], z['top'], z['bottom']],
            fill='toself', fillcolor='rgba(0,0,0,0)',
            line=dict(color='rgba(0,0,0,0)'),
            name=str(zone_id), showlegend=False,
            hovertemplate=hover_txt,
        ), row=1, col=1)

        fig.add_annotation(
            xref='x', yref='y', x=x0, y=float(z['midpoint']),
            text=f' {zone_id}  p={prob:.2f}',
            showarrow=False, xanchor='left',
            font=dict(size=8, color=line),
        )

    # Holiday rangebreaks
    all_days    = pd.date_range(df.index.min(), df.index.max(), freq='D')
    trading_set = set(df.index.normalize())
    holidays    = [d.strftime('%Y-%m-%d') for d in all_days
                   if d not in trading_set and d.weekday() < 5]
    range_breaks = [dict(bounds=['sat','mon']), dict(values=holidays)]

    fig.update_layout(
        title=dict(text=title or f'<b>{SYMBOL}</b> — ML-Detected Zones',
                   font=dict(size=14, color='#d1d4dc'),
                   x=0.0, xanchor='left', y=0.98, yanchor='top'),
        template=CHART_THEME, paper_bgcolor=PAPER_BG, plot_bgcolor=PLOT_BG,
        height=710, margin=dict(l=65, r=25, t=50, b=70),
        legend=dict(orientation='h', yanchor='top', y=-0.06, xanchor='right', x=1.0,
                    font=dict(size=10, color='#d1d4dc'), bgcolor='rgba(0,0,0,0)'),
        xaxis_rangeslider_visible=False,
        hovermode='x unified',
        hoverlabel=dict(bgcolor='#1e222d', font_size=11),
    )
    fig.update_xaxes(
        rangeselector=dict(
            buttons=[
                dict(count=1,  label='1M', step='month', stepmode='backward'),
                dict(count=3,  label='3M', step='month', stepmode='backward'),
                dict(count=6,  label='6M', step='month', stepmode='backward'),
                dict(count=1,  label='1Y', step='year',  stepmode='backward'),
                dict(count=2,  label='2Y', step='year',  stepmode='backward'),
                dict(step='all', label='ALL'),
            ],
            bgcolor='#1e222d', activecolor='#2196F3',
            font=dict(color='#d1d4dc', size=10),
            x=0.0, xanchor='left', y=-0.12, yanchor='top',
        ),
        showgrid=True, gridcolor=GRID_COLOR, zeroline=False,
        rangebreaks=range_breaks, row=1, col=1,
    )
    fig.update_xaxes(showgrid=True, gridcolor=GRID_COLOR,
                     rangebreaks=range_breaks, row=2, col=1)
    fig.update_yaxes(title_text='Price (INR)', gridcolor=GRID_COLOR,
                     tickformat=',.0f', side='right', row=1, col=1)
    fig.update_yaxes(title_text='Volume', gridcolor=GRID_COLOR,
                     tickformat='.2s', side='right', row=2, col=1)
    return fig


In [281]:
# ── All ML zones on full price history ───────────────────────────────────────
fig_full = plot_zones_on_candles(
    proc_idx, ml_zones_idx,
    title=f'<b>{SYMBOL}</b>  —  ML-Detected Zones  |  Full history  ({len(ml_zones_idx)} zones)',
)
fig_full.show()


---
## 6 · Individual ML Zone Deep Dives

Each ML-detected zone is plotted individually — arrival leg → base → departure → reaction.
- Dashed vertical line = base start
- Solid vertical line = zone formation (departure candle close)
- Zone band shaded green (demand) or red (supply)
- Volume + ATR in lower panel


In [282]:
def plot_zone_detail(
    proc,
    zone,
    zid,
    context=25,
):
    """
    Broker-style zoom-in on a single ML-detected zone.
    Identical to notebook 02 — adapted for ML zones (ml_prob replaces strength).
    """
    base_start = pd.Timestamp(zone['base_start_date'])
    base_end   = pd.Timestamp(zone['base_end_date'])
    dep_date   = pd.Timestamp(zone['formation_date'])

    try:
        base_pos = proc.index.get_loc(base_start)
    except KeyError:
        base_pos = int(proc.index.searchsorted(base_start))
    try:
        base_end_pos = proc.index.get_loc(base_end)
    except KeyError:
        base_end_pos = int(proc.index.searchsorted(base_end))
    try:
        dep_pos = proc.index.get_loc(dep_date)
    except KeyError:
        dep_pos = int(proc.index.searchsorted(dep_date))

    # Show at most 3 candles between Base start and Zone formed.
    # Cap relative to dep_pos (not base_end_pos) so gap candles don't add up.
    MAX_VIS_BASE = 3
    vis_base_start_pos = max(base_pos, dep_pos - MAX_VIS_BASE)

    lo     = max(0, vis_base_start_pos - context)
    hi     = min(len(proc), dep_pos + context + 1)
    window = proc.iloc[lo:hi].copy()
    base_start = proc.index[vis_base_start_pos]

    is_demand = zone['type'] == 'demand'
    fill  = DEMAND_FILL if is_demand else SUPPLY_FILL
    line  = DEMAND_LINE if is_demand else SUPPLY_LINE
    ztype = 'Demand (DBR — Drop → Base → Rally)' if is_demand \
            else 'Supply (RBD — Rally → Base → Drop)'

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.78, 0.22], vertical_spacing=0.02,
    )

    fig.add_trace(go.Candlestick(
        x=window.index,
        open=window['Open'], high=window['High'],
        low=window['Low'],   close=window['Close'],
        name='Price',
        increasing=dict(line=dict(color='#26a69a', width=1), fillcolor='#26a69a'),
        decreasing=dict(line=dict(color='#ef5350', width=1), fillcolor='#ef5350'),
        whiskerwidth=0.5,
    ), row=1, col=1)

    vol_colors = ['#26a69a' if c >= o else '#ef5350'
                  for c, o in zip(window['Close'], window['Open'])]
    fig.add_trace(go.Bar(
        x=window.index, y=window['Volume'].replace(0, None),
        marker_color=vol_colors, showlegend=False, opacity=0.55,
        hovertemplate='Vol: %{y:,.0f}<extra></extra>',
    ), row=2, col=1)

    if 'ATR' in window.columns:
        fig.add_trace(go.Scatter(
            x=window.index, y=window['ATR'],
            name='ATR', line=dict(color='#ff9800', width=1.2, dash='dot'),
            hovertemplate='ATR: %{y:.2f}<extra></extra>',
        ), row=2, col=1)

    # Zone rectangle
    fig.add_shape(
        type='rect', xref='x', yref='y',
        x0=window.index[0], x1=window.index[-1],
        y0=float(zone['bottom']), y1=float(zone['top']),
        fillcolor=fill, line=dict(color=line, width=1.5),
        layer='below',
    )

    # Top / bottom price labels
    for y_val, label in [(zone['top'], 'Zone Top'), (zone['bottom'], 'Zone Bot')]:
        fig.add_hline(
            y=float(y_val), line_color=line, line_width=1.2, line_dash='dash',
            annotation_text=f' {label} {y_val:.2f}',
            annotation_font_color=line, annotation_font_size=9,
            row=1, col=1,
        )

    # Base start vertical (dashed)
    if base_start in window.index:
        fig.add_shape(
            type='line', xref='x', yref='paper',
            x0=base_start, x1=base_start, y0=0, y1=1,
            line=dict(color='#aaaaaa', width=1.2, dash='dot'),
        )
        fig.add_annotation(
            xref='x', yref='paper', x=base_start, y=0.98,
            text=' Base start', showarrow=False, xanchor='left',
            font=dict(size=9, color='#aaaaaa'),
        )

    # Formation vertical (solid)
    if dep_date in window.index:
        dep_color = '#26a69a' if is_demand else '#ef5350'
        fig.add_shape(
            type='line', xref='x', yref='paper',
            x0=dep_date, x1=dep_date, y0=0, y1=1,
            line=dict(color=dep_color, width=1.8, dash='solid'),
        )
        fig.add_annotation(
            xref='x', yref='paper', x=dep_date, y=0.92,
            text=' Zone formed', showarrow=False, xanchor='left',
            font=dict(size=9, color=dep_color),
        )

    # Holiday rangebreaks
    all_days_w    = pd.date_range(window.index.min(), window.index.max(), freq='D')
    trading_set_w = set(window.index.normalize())
    holidays_w    = [d.strftime('%Y-%m-%d') for d in all_days_w
                     if d not in trading_set_w and d.weekday() < 5]
    range_breaks_w = [dict(bounds=['sat','mon']), dict(values=holidays_w)]

    prob  = zone.get('ml_prob', float('nan'))
    w_atr = zone.get('width_atr', zone['width'] / max(zone.get('avg_atr', 1), 1e-9))

    fig.update_layout(
        title=dict(
            text=(
                f'<b>{zid}</b>  {ztype}<br>'
                f'<sub>Zone: {zone["bottom"]:.2f} – {zone["top"]:.2f}  '
                f'| Width: {w_atr:.2f}× ATR  '
                f'| ML Prob: <b>{prob:.3f}</b>  '
                f'| Status: <b>{zone.get("status", "active")}</b>  '
                f'| Tests: {int(zone.get("test_count", 0))}  '
                f'| Base: {int(zone.get("base_length", 1))} candle(s)</sub>'
            ),
            font=dict(size=12, color='#d1d4dc'),
            x=0.0, xanchor='left',
        ),
        template=CHART_THEME, paper_bgcolor=PAPER_BG, plot_bgcolor=PLOT_BG,
        height=540, margin=dict(l=65, r=25, t=75, b=50),
        xaxis_rangeslider_visible=False, hovermode='x unified',
        hoverlabel=dict(bgcolor='#1e222d', font_size=10),
        legend=dict(font=dict(size=9), bgcolor='rgba(0,0,0,0)',
                    orientation='h', yanchor='top', y=-0.08, xanchor='right', x=1.0),
        showlegend=True,
    )
    fig.update_xaxes(gridcolor=GRID_COLOR, zeroline=False, rangebreaks=range_breaks_w)
    fig.update_yaxes(gridcolor=GRID_COLOR, tickformat=',.0f', side='right', row=1, col=1)
    fig.update_yaxes(gridcolor=GRID_COLOR, side='right', row=2, col=1)
    return fig


In [283]:
# ── Plot every ML-detected zone individually ─────────────────────────────────
print(f'Plotting {len(ml_zones_idx)} ML zones individually...\n')
for zid, zone in ml_zones_idx.iterrows():
    fig = plot_zone_detail(proc_idx, zone, str(zid), context=25)
    fig.show()


Plotting 96 ML zones individually...

